In [1]:
import re
import copy
import torch
import pandas as pd 

In [2]:
torch.manual_seed(42)

In [3]:
data=pd.read_csv('football_qa_dataset_full.csv')

In [4]:
data

,question,answer
0,Who won the 1930 FIFA World Cup?,Uruguay
1,Which team was crowned champion of the 1930 FI...,Uruguay
2,Which country hosted the 1930 FIFA World Cup?,Uruguay
3,In which country was the 1930 FIFA World Cup h...,Uruguay
4,Who was the runner-up in the 1930 FIFA World Cup?,Argentina
...,...,...
2877,Which MLS club did Lionel Messi join in 2023?,Inter-Miami
2878,Which club signed Zinedine Zidane from Juventu...,Real-Madrid
2879,Which club signed Luis Figo directly from riva...,Real-Madrid
2880,Which club signed Diego Maradona from Barcelon...,Napoli


In [5]:
#tokenize

def tokenize(text):
    text=text.lower()
    # drop punctuation (?, commas, full stops, apostrophes...) but keep hyphens used in names like inter-miami
    text=re.sub(r"[^\w\s-]","",text)
    
    return text.split()

In [6]:
tokenize('Who won the 1930 FIFA World Cup?')

['who', 'won', 'the', '1930', 'fifa', 'world', 'cup']

In [7]:
#train / validation / test split
# test is only used at the very end; validation is used for early stopping
train_val_data=data.sample(frac=0.8,random_state=42)
test_data=data.drop(train_val_data.index)

val_data=train_val_data.sample(frac=0.15,random_state=42)
train_data=train_val_data.drop(val_data.index)

print(len(train_data),len(val_data),len(test_data))

1960 346 576


In [8]:
#vocab (questions only, built from training data so val/test words can be unseen -> <UNK>)
vocab={'<PAD>':0,'<UNK>':1}

for question in train_data['question']:
    for token in tokenize(question):
        if token not in vocab:
            vocab[token]=len(vocab)

print(len(vocab))

1064


In [9]:
#answers are predicted as classes, so the model only chooses between real answers instead of every word
answers=sorted(data['answer'].unique())
answer_to_index={answer:i for i,answer in enumerate(answers)}

print(len(answers))

839


In [10]:
def text_to_indexs(text,vocab):
    indexed_text=[]
    
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])
        
    return indexed_text

In [11]:
text_to_indexs('Which country hosted the 1930 FIFA World Cup?',vocab)

[2, 28, 222, 5, 673, 33, 34, 7]

In [12]:
from torch.utils.data import Dataset,DataLoader
from torch.nn.utils.rnn import pad_sequence,pack_padded_sequence

In [13]:
class QADataset(Dataset):
    def __init__(self,data,vocab):
        self.data=data
        self.vocab=vocab
        
    
    def __len__(self):
        return self.data.shape[0]
    
    def __getitem__(self, index):
        numerical_question=text_to_indexs(self.data.iloc[index]['question'],self.vocab)
        answer_index=answer_to_index[self.data.iloc[index]['answer']]
        
        return torch.tensor(numerical_question),torch.tensor(answer_index)

In [14]:
#pad questions in a batch to the same length so we can use batch_size > 1
def collate_fn(batch):
    questions,answers=zip(*batch)
    lengths=torch.tensor([len(q) for q in questions])
    padded_questions=pad_sequence(questions,batch_first=True,padding_value=vocab['<PAD>'])
    
    return padded_questions,lengths,torch.stack(answers)

In [15]:
train_dataset=QADataset(train_data,vocab)
val_dataset=QADataset(val_data,vocab)
test_dataset=QADataset(test_data,vocab)

In [16]:
train_dataset[2]

(tensor([12,  4,  5, 16, 17, 18, 19,  5, 20, 21, 22,  5, 23, 24]), tensor(453))

In [17]:
batch_size=32

train_loader=DataLoader(train_dataset,batch_size=batch_size,shuffle=True,collate_fn=collate_fn)
val_loader=DataLoader(val_dataset,batch_size=batch_size,collate_fn=collate_fn)
test_loader=DataLoader(test_dataset,batch_size=batch_size,collate_fn=collate_fn)

In [18]:
question,lengths,answer = next(iter(train_loader))
print(question.shape,lengths[:5],answer[:5])

torch.Size([32, 17]) tensor([14,  8, 14,  9,  7]) tensor([  0,  84, 426, 220, 669])


In [19]:
import torch.nn as nn

In [20]:
class LSTMModel(nn.Module):
    def __init__(self,vocab_size,num_answers,embedding_dim=128,hidden_dim=128,dropout=0.4):
        super().__init__()
        
        self.embedding=nn.Embedding(vocab_size,embedding_dim,padding_idx=vocab['<PAD>'])
        self.lstm=nn.LSTM(embedding_dim,hidden_dim,batch_first=True,bidirectional=True)
        self.dropout=nn.Dropout(dropout)
        self.fc=nn.Linear(2*hidden_dim,num_answers)
        
    def forward(self,question,lengths):
        embedded_question=self.dropout(self.embedding(question))
        # packing makes the LSTM skip the padding, so the final hidden state is from the real last word
        packed=pack_padded_sequence(embedded_question,lengths,batch_first=True,enforce_sorted=False)
        _,(hidden,_)=self.lstm(packed)
        final=torch.cat([hidden[0],hidden[1]],dim=1) #forward + backward direction
        output=self.fc(self.dropout(final))
        
        return output

In [21]:
epochs=100
patience=10 #stop if validation accuracy hasn't improved for this many epochs

In [22]:
model=LSTMModel(len(vocab),len(answers))

In [23]:
criterion=nn.CrossEntropyLoss()

optimizer=torch.optim.Adam(model.parameters(),lr=1e-3, weight_decay=1e-5)

In [24]:
def evaluate(model,loader):
    model.eval()
    correct=0
    total=0
    with torch.no_grad():
        for question,lengths,answer in loader:
            output=model(question,lengths)
            correct+=(output.argmax(dim=1)==answer).sum().item()
            total+=len(answer)
    return correct/total

In [25]:
#training loop with early stopping on validation accuracy

best_val_acc=0
best_state=None
epochs_without_improvement=0

for epoch in range(epochs):
    model.train()
    total_loss=0
    for question,lengths,answer in train_loader:
        optimizer.zero_grad()
        output=model(question,lengths)
        loss=criterion(output,answer)
        
        loss.backward()
        
        optimizer.step()
        
        total_loss=total_loss+loss.item()
    avg_loss=total_loss/len(train_loader)
    train_acc=evaluate(model,train_loader)
    val_acc=evaluate(model,val_loader)
    print(f'Epoch {epoch +1},loss:{avg_loss:4f},train acc:{train_acc:.3f},val acc:{val_acc:.3f}')
    
    if val_acc>best_val_acc:
        best_val_acc=val_acc
        best_state=copy.deepcopy(model.state_dict())
        epochs_without_improvement=0
    else:
        epochs_without_improvement+=1
        if epochs_without_improvement>=patience:
            print(f'Early stopping, best val acc:{best_val_acc:.3f}')
            break

model.load_state_dict(best_state)

Epoch 1,loss:6.266320,train acc:0.061,val acc:0.049


Epoch 2,loss:5.300845,train acc:0.105,val acc:0.092


Epoch 3,loss:4.730106,train acc:0.147,val acc:0.110


Epoch 4,loss:4.276691,train acc:0.203,val acc:0.104


Epoch 5,loss:3.914852,train acc:0.229,val acc:0.133


...

Epoch 27,loss:1.291834,train acc:0.876,val acc:0.179


Epoch 28,loss:1.225392,train acc:0.897,val acc:0.179


Epoch 29,loss:1.192693,train acc:0.913,val acc:0.173


Epoch 30,loss:1.140861,train acc:0.926,val acc:0.171
Early stopping, best val acc:0.199


<All keys matched successfully>

In [26]:
print(f'train acc:{evaluate(model,train_loader):.3f}')
print(f'val acc:{evaluate(model,val_loader):.3f}')
print(f'test acc:{evaluate(model,test_loader):.3f}')

train acc:0.730
val acc:0.199


test acc:0.146


In [27]:
def predict(model,question,threshold=0.3):
    model.eval()
    
    #covert question to numbers
    numerical_question=text_to_indexs(question,vocab)
    
    #tensor
    question_tensor=torch.tensor(numerical_question).unsqueeze(0)
    lengths=torch.tensor([len(numerical_question)])
    
    #send to model
    with torch.no_grad():
        output=model(question_tensor,lengths)
    
    #convert logits to proba 
    proba=torch.nn.functional.softmax(output,dim=1)
    
    #find max proba 
    value, index= torch.max(proba,dim=1)
    
    if value < threshold:
        print('idk')
    else:
        
        print(answers[index.item()])

In [28]:
predict(model,"team lost the final of 1954 FIFA World Cup?")

idk


In [29]:
predict(model,"Who won the 1954 FIFA World Cup?")

WestGermany


In [30]:
predict(model,"Which MLS club did Lionel Messi join in 2023?")

Inter-Miami
